# Sentiment analysis of kindle reviews using Word2Vec and RandomForest

##### Workflow
1. Preprocessing and clearning
2. Train test split
3. Word embeddings using Word2Vec
4. Training the model using RandomForest
5. testing the model

### Preprocessing and cleaning

In [35]:
import pandas as pd
df = pd.read_csv('all_kindle_review.csv')
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [36]:
df['reviewText'] = df['reviewText'] + df['summary']
df = df[['rating','reviewText']]

In [37]:
df.head()

,rating,reviewText
0,3,"Jace Rankin may be short, but he's nothing to ..."
1,5,Great short read. I didn't want to put it dow...
2,3,I'll start by saying this is the first of four...
3,3,Aggie is Angela Lansbury who carries pocketboo...
4,4,I did not expect this type of book to be in li...


In [38]:
df.dropna(subset=['reviewText'], inplace=True)

In [39]:
from nltk import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [ ]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
from nltk.corpus import stopwords
lemmatizer = WordNetLemmatizer()
corpus = []

for i in range(0, len(df)):
    X = re.sub('[^a-zA-Z]', ' ', df['reviewText'].iloc[i])
    X = X.lower()
    X = X.split()
    X = [lemmatizer.lemmatize(word) for word in X if not word in set(stopwords.words('english'))]
    X = ' '.join(X)
    corpus.append(X)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\goggl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['jace rankin may short nothing mess man hauled saloon undertaker know famous bounty hunter oregon shot man saloon finished year long quest avenge sister murder trying figure next snotty nosed farm boy rescued gang bully offer money kill man forced ranch reluctantly agrees bring man justice kill outright first need tell sister widower news kyla kyle springer bailey riding trail sleeping ground past month trying find jace want revenge man killed husband took ranch amongst crime keen detour jace want take realizes option hide behind boy persona best try keep pace confrontation along way get shot jace discovers kyle kyla come clean whole reason need scoundrel dead hope still help book share touching moment slow blooming romance kyla find good reason fear men hide behind boy persona watching jace slowly pull shell help conquer fear endearing pain real deeply rooted disappear face sexiness neither understandable aversion marriage magically disappear round nookie would man drifted town town 

### Train Test Split

In [41]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    corpus,
    df['rating'],
    random_state=67,
    test_size=0.25
)

### Word embeddings

In [42]:
from gensim.models import Word2Vec
w2v = Word2Vec(
    sentences=x_train,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)
w2v_test = Word2Vec(
    sentences=x_test,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [43]:
import numpy as np
sentenc_vectors = []

for sent in x_train:
    vectors = []
    for word in sent:
        if word in w2v.wv:
            vectors.append(w2v.wv[word])
    if len(vectors) != 0:
        sentenc_vectors.append(np.mean(vectors, axis=0))
    else:
        sentenc_vectors.append(np.zeros(100))

In [44]:
len(sentenc_vectors)

8998

In [46]:
len(y_train)

8998

### Training the model

In [61]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()

In [62]:
classifier.fit(sentenc_vectors, y_train)

RandomForestClassifier()

### Testing the model

In [49]:
import numpy as np
x_test_vec = []

for sent in x_test:
    vectors = []
    for word in sent:
        if word in w2v.wv:
            vectors.append(w2v_test.wv[word])
    if len(vectors) != 0:
        x_test_vec.append(np.mean(vectors, axis=0))
    else:
        x_test_vec.append(np.zeros(100))

In [63]:
y_pred = classifier.predict(x_test_vec)

In [51]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [64]:
accuracy_score(y_pred=y_pred, y_true=y_test)

0.171

In [65]:
print(confusion_matrix(y_pred, y_test))

[[364 391 387 581 609]
 [  4   0   4   3   6]
 [  1   1   0   0   1]
 [ 61  55  49  77 119]
 [ 50  44  44  77  72]]


In [66]:
print(classification_report(y_pred, y_test))

              precision    recall  f1-score   support

           1       0.76      0.16      0.26      2332
           2       0.00      0.00      0.00        17
           3       0.00      0.00      0.00         3
           4       0.10      0.21      0.14       361
           5       0.09      0.25      0.13       287

    accuracy                           0.17      3000
   macro avg       0.19      0.12      0.11      3000
weighted avg       0.61      0.17      0.23      3000



In [55]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(max_iter=1000)

In [59]:
classifier.fit(sentenc_vectors, y_train)

LogisticRegression(max_iter=1000)

In [60]:
accuracy_score(y_pred=y_pred, y_true=y_test)

0.187